# 00 · Descarga del DENUE (solo Querétaro)

El sistema recomienda **tipos de restaurante**, pero se descargan **todos los sectores**: oficinas, escuelas, consultorios y comercios alrededor son variables de contexto del modelo.

El INEGI publica cada edición del DENUE **nacional y partida por sector** (25 zips, unos 400–540 MB por edición). Este notebook, para cada edición:

1. Baja los 25 zips a una carpeta temporal **fuera de Dropbox**.
2. Lee cada CSV por bloques y se queda solo con Querétaro (`cve_ent = 22`).
3. Borra el zip y guarda el resultado en `data/raw/denue_qro_AAAA-MM.parquet` (unos pocos MB).

Si una edición ya está descargada, la salta. Calcula unos minutos por edición, según tu internet.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import io, re, tempfile, time, zipfile, urllib.request
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data/raw"
RAW.mkdir(parents=True, exist_ok=True)
TMP = Path(tempfile.gettempdir()) / "denue_tmp"     # fuera de Dropbox: no sincroniza los zips
TMP.mkdir(exist_ok=True)
BASE_URL = "https://www.inegi.org.mx/contenidos/masiva/denue"

## 1. Ediciones a descargar

Etiqueta `AAAA-MM` → carpeta del INEGI. La edición vigente (05/2026) no tiene carpeta: va con `None`.

Ediciones disponibles desde 2020: `2020_04`, `2020_11`, `2021_05`, `2021_11`, `2022_05`, `2022_11`, `2023_11` (no hay 05/2023), `2024_05`, `2024_11`, `2025_05` y la vigente.

Abajo están los 7 cortes anuales de la tesis.

In [ ]:
EDICIONES_DESCARGA = {
    "2020-04": "2020_04",
    "2021-05": "2021_05",
    "2022-05": "2022_05",
    "2023-11": "2023_11",     # no hay 05/2023
    "2024-05": "2024_05",
    "2025-05": "2025_05",
    "2026-05": None,          # edición vigente
}

# Los 25 archivos por sector que publica el INEGI desde 11/2019
SECTORES = ["11", "21", "22", "23", "31-33", "43", "46111", "46112-46311", "46321-46531",
            "46591-46911", "48-49", "51", "52", "53", "54", "55", "56", "61", "62",
            "71", "72_1", "72_2", "81_1", "81_2", "93"]

## 2. Funciones

- `url_archivo`: arma la liga. Las ediciones históricas llevan carpeta y sufijo `MMAA`, por ejemplo `2024_05/denue_00_62_0524_csv.zip`.
- `detectar_encoding`: las ediciones viejas vienen en `latin-1` y las nuevas en `utf-8`.
- `filtrar_queretaro`: lee el CSV por bloques de 200 mil filas para no llenar la memoria.

In [ ]:
def url_archivo(carpeta, sector):
    if carpeta is None:
        return f"{BASE_URL}/denue_00_{sector}_csv.zip"
    aaaa, mm = carpeta.split("_")
    return f"{BASE_URL}/{carpeta}/denue_00_{sector}_{mm}{aaaa[2:]}_csv.zip"

def descargar(url, destino, intentos=3):
    for i in range(intentos):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r, open(destino, "wb") as f:
                while bloque := r.read(1 << 20):
                    f.write(bloque)
            return
        except Exception as e:
            print(f"    intento {i + 1} falló: {e}")
            time.sleep(5)
    raise RuntimeError(f"No se pudo descargar {url}")

def detectar_encoding(z, info):
    with z.open(info) as f:
        muestra = f.read(2_000_000)
    muestra = muestra[: muestra.rfind(b"\n")]
    try:
        muestra.decode("utf-8")
        return "utf-8"
    except UnicodeDecodeError:
        return "latin-1"

def filtrar_queretaro(ruta_zip):
    with zipfile.ZipFile(ruta_zip) as z:
        csvs = [i for i in z.infolist() if i.filename.lower().endswith(".csv")]
        info = max(csvs, key=lambda i: i.file_size)          # el conjunto de datos, no el diccionario
        enc = detectar_encoding(z, info)
        partes = []
        with z.open(info) as f:
            lector = pd.read_csv(io.TextIOWrapper(f, encoding=enc), dtype=str, chunksize=200_000)
            for bloque in lector:
                bloque.columns = bloque.columns.str.strip().str.lower()
                partes.append(bloque[pd.to_numeric(bloque["cve_ent"], errors="coerce") == 22])
    return pd.concat(partes, ignore_index=True)

## 3. Descargar y filtrar

In [ ]:
for etiqueta, carpeta in EDICIONES_DESCARGA.items():
    salida = RAW / f"denue_qro_{etiqueta}.parquet"
    if salida.exists():
        print(f"{etiqueta}: ya existe, se salta")
        continue
    print(f"=== {etiqueta} ===")
    partes = []
    for s in SECTORES:
        url = url_archivo(carpeta, s)
        tmp = TMP / url.split("/")[-1]
        t0 = time.time()
        descargar(url, tmp)
        df = filtrar_queretaro(tmp)
        tmp.unlink()                                           # borra el zip
        partes.append(df)
        print(f"  {s:>12}: {len(df):>7,} registros de Querétaro ({time.time() - t0:.0f} s)")
    qro = pd.concat(partes, ignore_index=True)
    qro.to_parquet(salida, index=False)
    print(f"  → {salida.name}: {len(qro):,} registros\n")

## 4. Verificación

In [ ]:
for p in sorted(RAW.glob("denue_qro_*.parquet")):
    df = pd.read_parquet(p, columns=["cve_mun", "codigo_act"])
    print(f"{p.name}: {len(df):,} registros · {df['cve_mun'].nunique()} municipios · {p.stat().st_size / 1e6:.1f} MB")